# Smart Microscopy v3.2 - Adaptive Target Acquisition

Acquire an overview scan at the source objective, segment cells, pick targets from the cell distribution, then re-acquire those targets at the target objective.

## Workflow

| Step | What it does | Before you run it |
|------|--------------|-------------------|
| 1. Initialization | Connect, validate, boot the engine | Connect LAS X |
| 2a. Define limits | Derive XY stage limits | Place boundary markers |
| 2b. Define positions | Read the scan-field tiles | Draw the scan field |
| 2c. Define focus map | Fit a z-wide focus surface | Place focus markers |
| 2d. Archive & strip | Save the configured workflow, then strip LAS X for good | — |
| 3. Acquire overview | Scan and segment every tile | — |
| 4. Target discovery | Pick targets from the distribution | — |
| 5. Target acquisition | Re-acquire targets at high magnification | — |
| 6. Finish | Write the summary, plot, shut down | — |

This notebook is the operator UI. Workflow implementation lives in `pipeline/`; see `README.md` for the package layout.

## Configuration

Edit the cell below for your experiment, then run the steps in order.

`simulate=True` is a **simulator-only dry run**: every saved `.ome.tiff` from Step 3 and Step 5 has its pixels swapped for mock content (`mock_image_source` picks the source) under the original LAS X simulator's OME envelope, so the rest of the pipeline behaves identically to a real acquisition. A per-frame `SystemTypeName="SIMULATOR"` guard hard-aborts the run if any frame is not from the simulator — `simulate=True` cannot touch real-instrument data.

`visualize=False` skips the per-tile, per-target, and selection figures in Steps 3, 4, 5. Use it for large runs (e.g. well plates) where rendering and saving every frame becomes prohibitive; Step 2 plots always render.

**Before a real run:** set `simulate=False`, check each job's objective in LAS X, and set the z-galvo to 0.

In [ ]:
from _bootstrap import Config, Path

cfg = Config(
    # Jobs (each has an objective configured in LAS X)
    acquisition_job="Overview",
    target_job="HiRes",
    af_job="AF Job",
    # Paths
    analysis_repo=Path(r"Z:\zmbstaff\10374\Protocols_Notes\thom\notes\repositories\smart-analysis"),
    experiment="target-acquisition",  # output_root is derived from the LAS X AutoSave base folder as <autosave_base>/zmart
    # Simulator dry run (LAS X simulator only). When True, every saved
    # .ome.tiff from Step 3 / Step 5 has its pixels overwritten with
    # mock content; the OME envelope, naming, layout, and analysis flow
    # are otherwise unchanged. The per-frame SystemTypeName=="SIMULATOR"
    # guard makes this unsafe to enable on real hardware -- the run
    # hard-aborts on the first non-simulator frame.
    simulate=True,
    mock_image_source="skimage_human_mitosis",
    # Per-tile / per-target / selection visualization for Steps 3, 4, 5.
    # Set False for large runs (e.g. well plates) where rendering and
    # saving every tile/target is prohibitive. Step 2 plots are always
    # rendered.
    visualize=True,
)

## Step 1 — Initialization

Connects to LAS X, validates hardware and calibration, derives objective slots, and boots the analysis engine.

**Before running:** connect LAS X.

In [ ]:
from pipeline import connect_lasx, preflight

client = connect_lasx()
ctx = preflight(cfg, client)

## Step 2 — Setup overview

Configure the overview scan's spatial extent. Run 2a → 2b → 2c in order.

### Step 2a — Define limits

Derives XY stage limits from boundary markers (or uses the physical envelope if none are placed), writes the active limits record, and strips the template in place. Configure job z-mode in LAS X; this workflow does not modify the LRP.

**Before running:** place point markers at the sample corners in Navigator Expert, or skip to use the physical envelope.

In [ ]:
from pipeline import plot_stage_envelope, prepare_template

prepare_template(ctx)
plot_stage_envelope(ctx)

### Draw the scan field

Step 2a stripped the template's scan-field objects to keep LAS X responsive.
Now, in **Navigator Expert** (with the PythonInspect experiment still loaded):

1. Draw your scan field
2. Confirm the tile grid is visible *inside* the geometry, not just the outline
3. Run the next cell

If you drew the scan field before Step 2a, it was wiped. Just redraw it now.

### Step 2b — Define positions

Reads the scan field you drew — it defines which tiles the overview acquires.

**Before running:** draw the scan field in Navigator Expert.

In [ ]:
from pipeline import plot_scan_field, read_scan_field

read_scan_field(ctx)
plot_scan_field(ctx)

### Step 2c — Define focus map

Runs autofocus at each focus marker and fits a z-wide surface across the scan field.

**Before running:** place focus markers on the scan field in Navigator Expert.

In [ ]:
from pipeline import build_focus_map

focus_map = build_focus_map(ctx)
focus_map.plot(ctx)

### Step 2d — Archive pipeline and strip

Saves the configured pipeline (xml, lrp, rgn) to the initialization metadata folder, then strips the scan field and markers from Navigator Expert for the rest of the run. From here on the template stays stripped — Step 3 and Step 5 no longer touch the template.

In [ ]:
from pipeline import archive_and_strip

archive_and_strip(ctx)

## Step 3 — Acquire overview

Acquires every tile in snake order, segments each with cellpose, and writes the per-tile results to disk. Each tile is shown inline when `cfg.visualize=True`.

In [ ]:
from pipeline import run_overview

overview = run_overview(
    ctx,
    focus_map,
    live_display=cfg.visualize,
    save_png=cfg.visualize,
)

## Step 4 — Target discovery

Loads the overview, applies area / intensity thresholds, and picks target cells — shown as a scatter plus example crops.

Re-run with different `area_threshold`, `intensity_threshold`, `n_per_tile`, or `border_margin_px` to iterate without re-acquiring the overview.

In [ ]:
from pipeline.selection import load_overview_result, select_targets
from pipeline.visualize import display_selection

_analysis_dir = ctx.run.layout.analysis_dir("overview-scan")
overview = load_overview_result(_analysis_dir)  # kernel-restart safe

# Auto thresholds by default; uncomment to override. border_margin_px=64
# excludes cells whose bbox is within 64 px of any tile edge -- they have
# truncated stats and produce unreliable picks. Set to 0 to disable.
picks, selection = select_targets(
    overview,
    ctx.limits_context(),
    n_per_tile=4,
    border_margin_px=64,
    # area_threshold=200,
    # intensity_threshold=100,
    # seed=42,
)
if cfg.visualize:
    display_selection(
        selection,
        _analysis_dir,
        logs_dir=ctx.run.layout.logs_dir("overview-scan"),
    )

## Step 5 — Target acquisition

Switches to the target job and re-acquires each picked cell at high magnification; a failed pick does not stop the rest.

In [ ]:
from pipeline import acquire_targets

records = acquire_targets(
    ctx,
    picks,
    live_display=cfg.visualize,
    save_png=cfg.visualize,
)

## Step 6 — Finish

Writes `run_summary.json`, plots the results, and shuts down the engine.

The Cleanup cell below is a safety backstop for interrupted runs — not part of the normal path.

In [ ]:
from pipeline import finish, plot_results, write_summary
from pipeline.selection import load_overview_result

# Reload from disk so the finish cell is consistent with selection above
# (restart-between-selection-and-finish is NOT supported -- picks,
# selection, and records must still be in-kernel here).
overview = load_overview_result(ctx.run.layout.analysis_dir("overview-scan"))
write_summary(ctx, focus_map, overview, picks, selection, records)
plot_results(ctx, focus_map, picks, records)
finish(ctx)

In [ ]:
# Cleanup -- safety backstop for interrupted runs. Shuts down the
# analysis engine. Does NOT disconnect the LAS X client; it persists
# until kernel restart. Always safe to run.
try:
    ctx.shutdown()
except NameError:
    pass